<a href="https://colab.research.google.com/github/Sahil-K-Y/AI-ML-Blueprint/blob/main/Phase-6%20-%20Deep%20Learning%20%26%20PyTorch/102_pytorch_optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 102 — PyTorch Hyperparameter Search with Optuna

**Phase 6: Deep Learning & PyTorch**

## 📌 Topics & Learning Objectives
- Bayesian optimization vs grid/random search
- Setting up Optuna study and objective function
- Suggesting hyperparameters (learning rate, hidden units, optimizer)
- Early trial pruning with MedianPruner
- Extracting and visualizing best hyperparameters


### 💡 Concept Overview
Optuna uses Tree-structured Parzen Estimator (TPE) Bayesian sampling to discover optimal learning rates, architectures, and batch sizes in a fraction of the time required by grid search.

In [1]:
!pip install -q optuna
import torch
import torch.nn as nn
import optuna

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    hidden = trial.suggest_categorical("hidden", [16, 32, 64])
    model = nn.Sequential(nn.Linear(8, hidden), nn.ReLU(), nn.Linear(hidden, 2)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    X = torch.randn(100, 8, device=device)
    y = torch.randint(0, 2, (100,), device=device)

    for epoch in range(5):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        trial.report(loss.item(), epoch)
        if trial.should_prune(): raise optuna.exceptions.TrialPruned()
    return loss.item()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)
print("Best trial params:", study.best_params)


[I 2026-11-14 18:39:04,507] A new study created in memory with name: no-name-69904aa2-142a-4951-b4a3-20a9c33788c9
[I 2026-11-14 18:39:11,991] Trial 0 finished with value: 0.7056983709335327 and parameters: {'lr': 0.00010014150568930437, 'hidden': 32}. Best is trial 0 with value: 0.7056983709335327.
[I 2026-11-14 18:39:12,008] Trial 1 finished with value: 0.6800520420074463 and parameters: {'lr': 0.0008799469599985537, 'hidden': 64}. Best is trial 1 with value: 0.6800520420074463.
[I 2026-11-14 18:39:12,018] Trial 2 finished with value: 0.6177288889884949 and parameters: {'lr': 0.010665215786019278, 'hidden': 64}. Best is trial 2 with value: 0.6177288889884949.
[I 2026-11-14 18:39:12,029] Trial 3 finished with value: 0.6311551928520203 and parameters: {'lr': 0.008754219067270474, 'hidden': 64}. Best is trial 2 with value: 0.6177288889884949.
[I 2026-11-14 18:39:12,041] Trial 4 finished with value: 0.6987325549125671 and parameters: {'lr': 0.005779067391275746, 'hidden': 16}. Best is tri

Best trial params: {'lr': 0.010665215786019278, 'hidden': 64}


In [2]:
import optuna.visualization as vis

print("Best Loss Value :", study.best_value)
print("Trials Summary  :\n", study.trials_dataframe()[["number", "value", "params_lr", "params_hidden", "state"]])

# Optimization history chart
vis.plot_optimization_history(study).show()

Best Loss Value : 0.6177288889884949
Trials Summary  :
    number     value  params_lr  params_hidden     state
0       0  0.705698   0.000100             32  COMPLETE
1       1  0.680052   0.000880             64  COMPLETE
2       2  0.617729   0.010665             64  COMPLETE
3       3  0.631155   0.008754             64  COMPLETE
4       4  0.698733   0.005779             16  COMPLETE
